# HumanAI Detect - Veri Toplama (Colab)

**Amac:** `ai_raw` ve `ai_humanized` etiketli metinleri Colab T4/A100 GPU uzerinde
`Qwen/Qwen2.5-7B-Instruct` modeli ile uretmek.

**ONEMLI - bu bir YENIDEN URETIM calistirmasi (2026-07-04):** Onceki 3000/3000
uretimde (2026-06-30/07-01) ai_raw/ai_humanized ortalama ~234 kelime cikmisti,
insan verisi ise ~615 kelime -- bu uzunluk farki tek basina siniflari %71
macro-F1 ile ayirt edebiliyordu (ciddi confound, SHAP analiziyle dogrulandi).
Kod guncellendi: `llm_generators.py`/`humanizers.py` artik her ornek icin
insan dagilimina yakin (ort. 605, std 150, 200-1200 araliginda) rastgele bir
hedef kelime sayisi belirleyip prompt'a ekliyor, `max_new_tokens` 512'den
2800'e cikarildi. **Bu, uretimin onceki calistirmadan kabaca 2.5x daha uzun
surmesi anlamina gelir** (ortalama ~2.6x daha fazla token uretiliyor).

Fresh bir Colab klonunda `data/raw/` bos oldugu icin (`.gitignore`'da) script
otomatik olarak sifirdan, YENI kodla uretir -- ayrica bir temizlik gerekmez.

**Adimlar:**
1. GPU kontrolu
2. Repo'yu klonla, bagimliliklari kur
3. `ai_raw` uret (yeni uzunluk-hedefli kod)
4. `ai_humanized` uret (ai_raw'in uzunlugunu buyuk olcude korur, prompt geregi)
5. Sonuclari dogrudan Colab uzerinden tarayiciya indir (Drive kullanilmiyor)

In [ ]:
# 1. GPU kontrolü
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 3a. Repo klonla (GitHub URL'ini kendi repo'nla değiştir)
GITHUB_REPO = 'https://github.com/KULLANICI_ADINIZ/humanai-detect.git'  # <- BURAYA KENDİ REPO URL'İNİ YAZ
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect

In [ ]:
# 3b. Bağımlılıkları kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q

In [ ]:
# 3c. .env oluştur (API key gerekmez, transformers lokal çalışır)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 3d. target_count_per_class ayarla (test icin 50, gercek run icin 3000)
import yaml, re

TARGET = 3000  # <- istedigin sayiya ayarla (test icin 50, gercek run icin 3000)

with open('configs/data_sources.yaml', 'r', encoding='utf-8') as f:
    content = f.read()

content = re.sub(
    r'(target_count_per_class:\s*)\d+',
    f'target_count_per_class: {TARGET}',
    content
)
with open('configs/data_sources.yaml', 'w', encoding='utf-8') as f:
    f.write(content)

print(f'target_count_per_class = {TARGET}')

In [ ]:
# 4. ai_raw uret (Qwen2.5-7B-Instruct, 4-bit quantized) -- her ornek icin
# rastgele hedef kelime sayisi (ort. ~605, insan dagilimina yakin) prompta eklenir.
# Model ilk yuklemede ~5 dk surer. Onceki calistirmadan (~234 kelime/ornek)
# kabaca 2.5x daha uzun surmesi beklenir (~605 kelime/ornek hedefleniyor).
!python scripts/collect_data.py --label ai_raw

In [ ]:
# 4b. GUVENLIK: ai_raw bitince hemen indir. Boylece 5. adim (ai_humanized,
# daha uzun surer, oturum kopma riski daha yuksek) coker/kopar da
# /content sifirlanirsa, ai_raw'i yeniden URETMEK yerine bu dosyayi
# tekrar yukleyip devam edebilirsin (asagidaki not hucresine bak).
from pathlib import Path
from google.colab import files

src = Path('data/raw/ai_raw/ai_raw.jsonl')
if src.exists():
    lines = sum(1 for _ in open(src))
    print(f'ai_raw: {lines} ornek -> indiriliyor (yedek amacli)...')
    files.download(str(src))
else:
    print('ai_raw.jsonl bulunamadi!')

**Eger 5. adim (ai_humanized) sirasinda oturum koptu/coktu ve yeniden
klonlaman gerektiyse:** `ai_raw` uretimini BASTAN yapma. Onceki hucreden
indirdigin `ai_raw.jsonl`'i asagidaki hucreyle geri yukle, sonra 5. adima
gec.

In [ ]:
# 4c. (SADECE oturum koptuysa calistir) ai_raw.jsonl'i geri yukle
from pathlib import Path
from google.colab import files

Path('data/raw/ai_raw').mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # ai_raw.jsonl'i sec
for name in uploaded:
    Path(name).rename('data/raw/ai_raw/ai_raw.jsonl')
    print(f'{name} -> data/raw/ai_raw/ai_raw.jsonl olarak tasindi')

In [ ]:
# 5. ai_humanized uret -- humanize prompt'u orijinal (yeni, uzun) ai_raw
# metninin uzunlugunu buyuk olcude korumasini istiyor, bu yuzden ayrica bir
# hedef-uzunluk parametresi gerekmiyor.
!python scripts/collect_data.py --label ai_humanized

In [ ]:
# 6. Sonuçları doğrudan Colab üzerinden yerel makineye indir (Drive kullanılmıyor)
from pathlib import Path
from google.colab import files

for label in ['ai_raw', 'ai_humanized']:
    src = Path(f'data/raw/{label}/{label}.jsonl')
    if src.exists():
        lines = sum(1 for _ in open(src))
        print(f'{label}: {lines} ornek -> indiriliyor...')
        files.download(str(src))
    else:
        print(f'{label}: dosya bulunamadi!')

## İndirilen Dosyaları Yerleştirme

Önceki hücre `ai_raw.jsonl` ve `ai_humanized.jsonl`'i tarayıcının varsayılan
indirme klasörüne (`Downloads`) tek tek indirir (Colab bazı tarayıcılarda
"birden fazla dosya indirilsin mi?" onayı isteyebilir).

Colab bittikten sonra:
1. İndirilen `ai_raw.jsonl` ve `ai_humanized.jsonl` dosyalarını `Downloads` klasöründen al
2. Yerel projede şuraya koy (eski kısa-metin dosyalarının üzerine yaz):
   - `data/raw/ai_raw/ai_raw.jsonl`
   - `data/raw/ai_humanized/ai_humanized.jsonl`
3. Sonra `preprocess.py`, `build_features.py`, `train_model.py` sırasıyla çalıştır